# Arabic Cursive OCR Benchmark — Local Models, Capped at 20/dataset

A companion to `04_colab_sweep.ipynb` for filling in the gaps left by the
local models we've already tried (`qari-ocr-0.2`, `qari-ocr-0.3`,
`qwen2-vl-2b`, `qwen2.5-vl-3b`, `trocr-large-arabic`) across the datasets
they haven't fully covered yet. Unlike notebook 04 (which adds `BATCH` new
lines *every* re-run, forever), this one tops each (model, dataset) pair up
to **`TARGET` = 20 total scored lines and stops there** — re-running it is
always safe (already-at-20 pairs are skipped, nothing is ever re-spent),
but it will never grow past 20 on its own. Raise `TARGET` by hand for a
bigger pass later.

**Incremental + resumable:** same per-line cache as every other notebook —
already-scored lines are never recomputed, and a pair already at `TARGET`
is skipped with no empty run/commit.

**Before running:** `Runtime -> T4 GPU`; add `GH_TOKEN` (write-scoped) and
`HF_TOKEN` secrets (see `notebooks/README.md`).

## 1. GPU (falls back to CPU)

In [ ]:
import torch
HAS_GPU = torch.cuda.is_available()
if HAS_GPU:
    print('GPU:', torch.cuda.get_device_name(0))
    !nvidia-smi --query-gpu=memory.total,memory.used --format=csv
else:
    print('NO GPU — will run on CPU (much slower; big 4-bit models are skipped).')
    print('For a real run: Runtime -> Change runtime type -> T4 GPU.')

## 2. Repo

In [ ]:
# Bulletproof checkout: works whether the repo is missing, at the wrong
# path, or has local edits. Always ends up at the latest origin/main.
import os
REPO = '/content/arabic-ocr-benchmark'
token = None
try:
    from google.colab import userdata
    token = userdata.get('GH_TOKEN')
except Exception as e:
    print('no GH_TOKEN secret:', e)
url = f"https://{(token + '@') if token else ''}github.com/rabeeh97/arabic-ocr-benchmark.git"
if os.path.isdir(REPO + '/.git'):
    os.system(f'git -C {REPO} fetch --depth 1 origin main && git -C {REPO} reset --hard origin/main')
    # drop untracked leftovers (e.g. a stale sweep_manifest.json) so the
    # manifest is always exactly what's in the repo. Keeps gitignored _cache.
    os.system(f'git -C {REPO} clean -fd -- results')
else:
    os.system(f'rm -rf {REPO} && git clone --depth 1 {url} {REPO}')
os.chdir(REPO)
# src-layout package: importable straight from the checkout even if the
# `pip install -e` cell fails or is skipped (this is what caused the old
# 'No module named arabic_ocr_bench' in the sweep cell).
import sys
for _p in (os.path.join(os.getcwd(), 'src'), os.getcwd()):
    if _p not in sys.path: sys.path.insert(0, _p)
print(os.popen('git log --oneline -3').read())

## 3. Install + caches

In [ ]:
# Keep all downloads on Colab's own scratch disk (ephemeral, ~78 GB) --
# never the local machine. Nothing here touches your computer.
import os
os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
# Optional HF_TOKEN secret: lifts anonymous rate limits and speeds the
# Muharaf download (and is required for gated models like PaliGemma's base).
try:
    from google.colab import userdata
    _hf = userdata.get('HF_TOKEN')
    if _hf:
        os.environ['HF_TOKEN'] = _hf
        print('HF_TOKEN set')
except Exception:
    pass

In [ ]:
!pip install -q -e '.[local]'
!pip uninstall -y -q torchao 2>/dev/null; echo 'torchao removed (unused)'

## 4. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 5. Config

Only the local models already on the leaderboard — nothing untried
(`qwen2.5-vl-7b`, `ain-7b`, `minicpm-v` stay in notebook 04 for a later,
uncapped pass). `DATASETS` is every benchmark set; pairs already at
`TARGET` are skipped automatically, so it's safe to list ones some models
have already finished (e.g. `qari-ocr-0.2` on `muharaf_lines`).

In [ ]:
MODELS = [
    'qari-ocr-0.2', 'qari-ocr-0.3', 'qwen2-vl-2b', 'qwen2.5-vl-3b',
    'trocr-large-arabic',
]
DATASETS = [
    'kitab_khatt', 'kitab_muharaf', 'kitab_historicalbooks', 'kitab_hindawi',
    'kitab_historyar', 'kitab_onlinekhatt', 'muharaf_lines',
]
TARGET = 20              # cap per (model, dataset); already-scored lines count toward it

import yaml
_mc = yaml.safe_load(open('configs/models.yaml'))
if not HAS_GPU:
    import psutil
    _free = psutil.virtual_memory().available / 1e9
    _keep = []
    for m in MODELS:
        c = _mc.get(m, {})
        if c.get('load_in_4bit'):
            print(f'  drop {m}: 4-bit, GPU-only'); continue
        need = float(c.get('min_ram_gb', 0) or 0)
        if need and need > _free:
            print(f'  drop {m}: needs ~{need:.0f} GB, only {_free:.1f} GB free'); continue
        _keep.append(m)
    MODELS = _keep
    print(f'CPU mode ({_free:.1f} GB free) -> MODELS={MODELS}')

## 6. Run the capped sweep

Tops up every (model, dataset) pair to `TARGET` lines and stops. Safe to
Run all again — anything already at `TARGET` is skipped with no commit.

In [ ]:
import os, sys, shutil, subprocess, pathlib
for _p in (os.path.join(os.getcwd(), 'src'), os.getcwd()):
    if _p not in sys.path: sys.path.insert(0, _p)
from arabic_ocr_bench.sweep import run_sweep
from arabic_ocr_bench.runner import scored_indices
from arabic_ocr_bench.samples import export_run_samples

DRIVE = pathlib.Path('/content/drive/MyDrive/arabic-ocr-bench')
DRIVE_OK = DRIVE.parent.exists()   # is Drive mounted?

_DS = {}
_HF_SPLIT = {
    'muharaf_lines': ('aamijar/muharaf-public', 'test'),
    'kitab_khatt': ('ahmedheakl/arocrbench_khatt', 'train'),
    'kitab_muharaf': ('ahmedheakl/arocrbench_muharaf', 'train'),
    'kitab_historicalbooks': ('ahmedheakl/arocrbench_historicalbooks', 'train'),
    'kitab_hindawi': ('ahmedheakl/arocrbench_hindawi', 'train'),
    'kitab_historyar': ('ahmedheakl/arocrbench_historyar', 'train'),
    'kitab_onlinekhatt': ('ahmedheakl/arocrbench_onlinekhatt', 'train'),
}  # add datasets here (must match configs/datasets.yaml)
def _get_image_fn(dataset):
    if dataset not in _HF_SPLIT:
        return lambda i: None
    if dataset not in _DS:
        from datasets import load_dataset
        hf_id, split = _HF_SPLIT[dataset]
        _DS[dataset] = load_dataset(hf_id, split=split)
    ds = _DS[dataset]
    return lambda i: ds[int(i)]['image']

def _git(*a):
    return subprocess.run(['git', *a], capture_output=True, text=True)

def _mirror_to_drive(run_dir):
    if not DRIVE_OK:
        return
    (DRIVE / 'results').mkdir(parents=True, exist_ok=True)
    if pathlib.Path('results/_cache').is_dir():
        shutil.copytree('results/_cache', DRIVE / '_cache', dirs_exist_ok=True)
    shutil.copytree(run_dir, DRIVE / 'results' / pathlib.Path(run_dir).name, dirs_exist_ok=True)
    for f in ('sweep_manifest.json', 'leaderboard.html', 'leaderboard.json'):
        if pathlib.Path('results', f).is_file():
            shutil.copy2(pathlib.Path('results', f), DRIVE / 'results' / f)

def checkpoint(entry, run_dir):
    run_dir = str(run_dir)
    try:
        export_run_samples(run_dir, get_image=_get_image_fn(entry['dataset']), k=3)
    except Exception as e:
        print('   sample export failed:', e)
    subprocess.run(['python', 'scripts/make_leaderboard.py', 'results',
                    '-o', 'results/leaderboard.html'])
    _mirror_to_drive(run_dir)
    if not token:
        print('   no GH_TOKEN -> Drive only (this counts as persisted)')
        return
    _git('config', 'user.email', 'colab@arabic-ocr-bench')
    _git('config', 'user.name', 'colab-sweep')
    _git('add', '-f', run_dir, 'results/_cache', 'results/sweep_manifest.json',
         'results/leaderboard.html', 'results/leaderboard.json')
    _git('commit', '-m', f"sweep: {entry['model']} {entry['phase']} "
         f"CER-light {entry['cer_micro_light']*100:.1f}% n={entry['n']}")
    _git('pull', '--rebase', '--autostash', 'origin', 'main')
    p = _git('push', 'origin', 'HEAD:main')
    if p.returncode != 0:
        raise RuntimeError('git push failed: ' + (p.stderr.strip()[:300] or 'unknown'))
    print('   push: ok')

if DRIVE_OK and (DRIVE / '_cache').is_dir():
    shutil.copytree(DRIVE / '_cache', 'results/_cache', dirs_exist_ok=True)
    print('restored line cache from Drive')

# Capped sweep: each (model, dataset) pair is topped up to TARGET total scored
# lines, never more — unlike 04_colab_sweep.ipynb (which adds BATCH new lines
# every re-run, forever), a pair already at or past TARGET is skipped outright.
# Already-cached lines (from a prior 04 run, or an earlier call to this notebook)
# count toward TARGET, so nothing already scored is re-spent.
import gc
def _gpu_free_gb():
    try:
        import torch
        if torch.cuda.is_available():
            return torch.cuda.mem_get_info()[0] / 1e9
    except Exception:
        pass
    return None

def _reclaim():
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.synchronize(); torch.cuda.empty_cache()
    except Exception:
        pass
    gc.collect()

# Capped sweep: each (model, dataset) pair is topped up to TARGET total scored
# lines, never more — unlike 04_colab_sweep.ipynb (which adds BATCH new lines
# every re-run, forever), a pair already at or past TARGET is skipped outright.
# Already-cached lines (from a prior 04 run, or an earlier call to this notebook)
# count toward TARGET, so nothing already scored is re-spent.
#
# Between models: reclaim GPU memory (the runner already calls the client's
# close(), this is a second belt) and, before starting one, refuse it if there
# isn't enough free VRAM for its configs/models.yaml `min_ram_gb` budget — a
# leaked previous model would otherwise OOM-crash the runtime here.
RESULTS_DIR = pathlib.Path('results')
for _ds in DATASETS:
    print(f'\n########## DATASET: {_ds} ##########')
    for _m in MODELS:
        _done = scored_indices(RESULTS_DIR, _m, _ds)
        _remaining = max(0, TARGET - len(_done))
        if _remaining == 0:
            print(f'  skip {_m} / {_ds} (already has {len(_done)} >= TARGET={TARGET})')
            continue
        _reclaim()
        _need = float(_mc.get(_m, {}).get('min_ram_gb', 0) or 0)
        _free = _gpu_free_gb()
        if _need and _free is not None and _free < _need:
            print(f'  SKIP {_m} / {_ds}: needs ~{_need:.0f} GB VRAM, only {_free:.1f} GB free '
                  f'(a previous model did not release its memory). Re-run the notebook to retry it.')
            continue
        _fs = f' [{_free:.1f} GB VRAM free]' if _free is not None else ''
        print(f'  {_m} / {_ds}: {len(_done)} scored, adding up to {_remaining} more{_fs}')
        try:
            run_sweep(
                models=[_m], dataset=_ds,
                partial_limit=_remaining, run_full=False,
                accumulate=True,
                concurrency=1,
                manifest_path='results/sweep_manifest.json', results_dir='results',
                on_progress=print, on_done=checkpoint,
            )
        except Exception as e:
            import traceback; traceback.print_exc()
            print(f'!!! {_m} / {_ds} FAILED ({type(e).__name__}: {e}) — continuing !!!')
        finally:
            _reclaim()
print('\n=== CAPPED SWEEP COMPLETE ===')

## 7. Final leaderboard

In [ ]:
!python scripts/make_leaderboard.py results -o results/leaderboard.html
from IPython.display import HTML
HTML(open('results/leaderboard.html').read())

## Notes

- **This notebook never grows past `TARGET`.** Raise it and Run all again
  for a bigger pass — already-scored lines are never re-spent either way.
- **Uncapped, ever-growing sweep** (all 8 local models, incl. untried
  ones): `04_colab_sweep.ipynb`.
- **Hosted models (Gemini/GPT/Claude):** `02_colab_hosted_apis.ipynb`.